In [1]:
import tensorflow as tf
tf.config.set_visible_devices([], "GPU")   # using CPU 
import keras
print(tf.__version__, keras.__version__)
print(tf.config.get_visible_devices())

2.21.0 3.15.1
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [2]:
import numpy as np
from fftdeep.data.arrays import get_arrays
from fftdeep.models.cnn import build_cnn, compile_model

keras.utils.set_random_seed(42)
Xtr, ytr = get_arrays("train")
Xva, yva = get_arrays("val")
Xtr, Xva = Xtr.astype("float32"), Xva.astype("float32")   
ytr, yva = ytr.astype("float32"), yva.astype("float32")

model = compile_model(build_cnn())
model.summary()
print("A:", model.count_params())
print("B:", build_cnn(convs_per_block=2, batch_norm=True).count_params())

Model: "cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip (RandomFlip)        │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 93,377 (364.75 KB)

 Trainable params: 93,377 (364.75 KB)

 Non-trainable params: 0 (0.00 B)

A: 93377
B: 288481


In [3]:
print(model.evaluate(Xva[:2000], yva[:2000], verbose=0, return_dict=True))

{'accuracy': 0.5, 'auc': 0.4433389902114868, 'loss': 0.7057311534881592, 'precision': 0.5, 'recall': 1.0}


In [4]:
rng = np.random.default_rng(0)
idx = rng.choice(len(Xtr), 256, replace=False)
tiny = compile_model(build_cnn(augment=False, dropout=0.0))
h = tiny.fit(Xtr[idx], ytr[idx], epochs=100, batch_size=32, verbose=0)
print(h.history["accuracy"][-1], h.history["loss"][-1])

0.9921875 0.04897824674844742


In [5]:
import time
m = compile_model(build_cnn())
for i in range(2):
    t = time.perf_counter()
    h = m.fit(Xtr[:20000], ytr[:20000], epochs=1, batch_size=128, verbose=0)
    print(f"run {i}: {time.perf_counter() - t:.1f}s  loss={h.history['loss'][0]:.4f}")

run 0: 5.7s  loss=0.5460
run 1: 4.5s  loss=0.4273


Time taken 
1. GPU - run 0: 14.0s  loss=0.5458
run 1: 13.6s  loss=0.4274

2. CPU - 